In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from scipy.spatial import distance

# Load YOLOv8 model
model_path = r"C:\Users\Kaushik\code_JN\runs\detect\train2\weights\best.pt"
model = YOLO(model_path)

# Open webcam
cap = cv2.VideoCapture(0)

previous_centroids = {}

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    current_centroids = {}

    for box in results[0].boxes.data.tolist():
        x1, y1, x2, y2, conf, cls_id = box
        if int(cls_id) == 0:  # 'person' class
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)  # Compute center point
            current_centroids[(cx, cy)] = None  # Store new detections

            # Draw bounding box and center
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)  # Make the center dot bigger

    # Match new centroids with previous frame
    for (cx, cy) in current_centroids.keys():
        closest = None
        min_dist = float("inf")

        for (px, py) in previous_centroids.keys():
            d = distance.euclidean((cx, cy), (px, py))
            if d < min_dist:
                min_dist = d
                closest = (px, py)

        if closest and min_dist < 100:  # Increase threshold to reduce noise
            dx, dy = cx - closest[0], cy - closest[1]  # Direction vector
            scale = 3  # Adjust this value to make arrows bigger
            arrow_end = (cx + dx * scale, cy + dy * scale)  # Scale the arrow length

            cv2.arrowedLine(frame, closest, (int(arrow_end[0]), int(arrow_end[1])), (0, 0, 255), 3, tipLength=0.3)

    previous_centroids = current_centroids.copy()  # Update for next frame

    # Display frame
    cv2.imshow("Centroid Tracking - Crowd Movement", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 1 people, 43.6ms
Speed: 6.3ms preprocess, 43.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 26.4ms
Speed: 1.5ms preprocess, 26.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 26.2ms
Speed: 0.0ms preprocess, 26.2ms inference, 3.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 25.0ms
Speed: 1.0ms preprocess, 25.0ms inference, 3.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 28.4ms
Speed: 0.0ms preprocess, 28.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 29.5ms
Speed: 0.0ms preprocess, 29.5ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 26.2ms
Speed: 1.0ms preprocess, 26.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 29.8ms
Speed: 1.0ms preprocess, 29.8ms inference, 0.0ms postprocess per image at shape (1, 3, 48

KeyboardInterrupt: 

In [20]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
from scipy.spatial import distance

# Load YOLOv8 model
model_path = r"C:\Users\Kaushik\code_JN\runs\detect\train2\weights\best.pt"
model = YOLO(model_path)

# Open webcam
cap = cv2.VideoCapture(0)

previous_centroids = {}
arrow_directions = {}  # Stores last known arrow direction
arrow_timestamps = {}  # Tracks last update time

arrow_delay = 0.7  # Delay in seconds (0.5s = 15 frames at ~30 FPS)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    current_centroids = {}

    for box in results[0].boxes.data.tolist():
        x1, y1, x2, y2, conf, cls_id = box
        if int(cls_id) == 0:  # 'person' class
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)  # Compute center point
            current_centroids[(cx, cy)] = None  # Store new detections

            # Draw bounding box and center
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)  # Make the center dot bigger

    # Match new centroids with previous frame
    for (cx, cy) in current_centroids.keys():
        closest = None
        min_dist = float("inf")

        for (px, py) in previous_centroids.keys():
            d = distance.euclidean((cx, cy), (px, py))
            if d < min_dist:
                min_dist = d
                closest = (px, py)

        if closest and min_dist < 100:  # Reduce noise
            dx, dy = cx - closest[0], cy - closest[1]  # Direction vector
            timestamp = time.time()

            # If no direction stored, or enough time has passed, update direction
            if closest not in arrow_directions or (timestamp - arrow_timestamps.get(closest, 0)) > arrow_delay:
                arrow_directions[closest] = (dx, dy)  # Store new direction
                arrow_timestamps[closest] = timestamp  # Update last update time

            # Draw the last confirmed arrow direction
            if closest in arrow_directions:
                dx, dy = arrow_directions[closest]
                scale = 5  # Adjust arrow size
                arrow_end = (closest[0] + dx * scale, closest[1] + dy * scale)
                cv2.arrowedLine(frame, closest, (int(arrow_end[0]), int(arrow_end[1])), (0, 0, 255), 3, tipLength=0.3)

    previous_centroids = current_centroids.copy()  # Update for next frame

    # Display frame
    cv2.imshow("Centroid Tracking - Stable Arrow Movement", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 42.8ms
Speed: 1.0ms preprocess, 42.8ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 33.6ms
Speed: 2.0ms preprocess, 33.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 30.8ms
Speed: 2.0ms preprocess, 30.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 29.9ms
Speed: 1.0ms preprocess, 29.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 31.0ms
Speed: 1.0ms preprocess, 31.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 30.9ms
Speed: 1.0ms preprocess, 30.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 32.6ms
Speed: 1.0ms preprocess, 32.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.7ms
Speed: 2.0ms preprocess, 29.7ms inference, 0.0ms postp

KeyboardInterrupt: 

In [21]:
cap.release()
cv2.destroyAllWindows()